## DSAN 6000 Homework 3A: Allocating Tasks to Parallel Workers with `joblib`

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience using **`joblib`** to quickly parallelize an **embarrassingly-parallel** task.

In this case, the problem is one you've likely already seen before, the basic **data cleaning** task of "normalizing" the format of a bunch of **phone numbers** that have been submitted by different users of a web form. This basic task was chosen specifically so that you can take your intuitions about how long it might take for a **serial** algorithm to complete, and compare with how quickly you'll be able to complete it by using `joblib` to **distribute** the subtasks to different **workers**, who can process the numbers in parallel.

In [1]:
import boto3

In [2]:
s3 = boto3.client('s3')

In [29]:
s3.download_file('dsan6000-data', 'form_submissions_5m.parquet', 'data/form_submissions_5m.parquet')

In [30]:
import pandas as pd

In [31]:
pnum_df = pd.read_parquet("data/form_submissions_5m.parquet")

In [32]:
pnum_df

,submitted,phone_number
0,2026-09-02 11:47:12.307260,(642)825-2065
1,2026-08-21 13:02:30.565568,(665)350-8504
2,2026-09-05 03:48:24.994327,831-693-9840
3,2026-09-03 10:16:07.950533,806.698.3098
4,2026-09-01 05:11:55.724196,(801)875-3909
...,...,...
4999995,2026-08-22 21:59:22.144394,263.624.7946
4999996,2026-09-04 08:25:35.896262,6934224745
4999997,2026-08-19 20:15:06.836764,001-309-467-4731
4999998,2026-09-05 11:05:59.214292,544.635.2396


In [33]:
import re

In [44]:
def clean_pnum_serial(pnum):
  digit_finder = re.compile(r'\d')
  return ''.join(digit_finder.findall(pnum)[-10:])

In [45]:
clean_pnum_serial('+1(281)-330-8004')

'2813308004'

In [46]:
pnums = pnum_df['phone_number'].to_list()
pnums[:20]

['(642)825-2065',
 '(665)350-8504',
 '831-693-9840',
 '806.698.3098',
 '(801)875-3909',
 '338-596-2158',
 '2465621283',
 '766.393.9776',
 '989.633.7547',
 '(938)587-6306',
 '+1-470-456-5243',
 '6044633926',
 '001-665-533-0265',
 '001-370-824-8483',
 '001-636-362-8274',
 '856-663-3134',
 '851.426.9171',
 '(323)421-7693',
 '5825973928',
 '936.926.4473']

In [56]:
len(pnums)

5000000

In [47]:
import time
disp_time = lambda start, end: print('{:.4f} s'.format(end - start))

In [48]:
serial_start = time.time()
pnums_cleaned_serial = [clean_pnum_serial(p) for p in pnums]
serial_end = time.time()
disp_time(serial_start, serial_end)

10.9298 s


## Part 2: Cleaning Numbers in Parallel

In [49]:
import joblib

In [51]:
joblib.cpu_count()

2

In [61]:
parallel_runner = joblib.Parallel(n_jobs=2, batch_size=10000)

In [62]:
par_start = time.time()
pnums_cleaned_parallel = parallel_runner(
  joblib.delayed(clean_pnum_serial)(p) for p in pnums
)
par_end = time.time()
disp_time(par_start, par_end)

61.3469 s


In [54]:
class NumCleaner:
  def __init__(self):
    self.digit_finder = re.compile(r'\d')

  def clean_pnum(self, pnum):
    return ''.join(self.digit_finder.findall(pnum)[-10:])


In [55]:
my_cleaner = NumCleaner()

par_start = time.time()
pnums_cleaned_parallel = parallel_runner(
  joblib.delayed(my_cleaner.clean_pnum)(p) for p in pnums
)
par_end = time.time()
disp_time(par_start, par_end)

110.9968 s
